# 03 — Experimental results
Compare B0, B1, E1, E2 and E3 on the fixed seed-42 held-out test set. Run after `python -m src.evaluate --condition ... --stage 2 --seed 42 --stress all`. No training or inference is performed here. Local evaluator JSON/CSV exports are the source of truth; missing results remain missing. Rotten is positive (1). Do not tune on these test results.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/splits/split_seed42.csv').is_file()), None)
if ROOT is None: raise FileNotFoundError('Open notebook inside fruit-illum-norm with its split manifest.')
sys.path.insert(0, str(ROOT))
MET = ROOT/'outputs/metrics'
OUT = ROOT/'outputs/figures/results'
OUT.mkdir(parents=True, exist_ok=True)
MODELS = ['B0','B1','E1','E2','E3']
STRESSES = ['normal','dark','bright','warm_cast']
NAMES = dict(zip(MODELS,['Original','Gaussian 5×5','Gray World','CLAHE','Gray World + CLAHE']))
SEED, STAGE = 42, 2
manifest = pd.read_csv(ROOT/'data/splits/split_seed42.csv')
test = manifest.query('split == "test"').copy()
assert len(test)==320 and test.path.is_unique and test.label.value_counts().to_dict()=={0:160,1:160}
assert manifest.groupby('sha256').split.nunique().max()==1
print(ROOT, '\nTest images:',len(test))
def stem(model, stress): return f'{model}_seed{SEED}_stage{STAGE}_{stress}_test'
def save(fig,name):
    fig.savefig(OUT/name, dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)


## 1. Load and validate metrics
Require matching experiment metadata, sample counts, class totals and confusion-matrix accuracy. Missing files are listed, not replaced with zeros.

In [ ]:
records, missing = [], []
for model in MODELS:
    for stress in STRESSES:
        path = MET/(stem(model,stress)+'_metrics.json')
        if not path.exists():
            missing.append((model,stress)); continue
        d=json.loads(path.read_text(encoding='utf-8'))
        assert (d['condition'],d['seed'],d['stage'],d['split'],d['stress'])==(model,SEED,STAGE,'test',stress), path
        cm=np.asarray(d['confusion_matrix'])
        assert cm.shape==(2,2) and (cm>=0).all() and np.array_equal(cm.sum(axis=1),[160,160]), path
        assert d['n_samples']==320 and d['n_correct']==np.trace(cm) and np.isclose(d['accuracy'],np.trace(cm)/320), path
        assert d['rotten_misclassified_as_fresh']==cm[1,0], path
        records.append(d)
print(f'Available: {len(records)}/20')
display(pd.DataFrame(missing, columns=['model','stress']))
if not records: raise FileNotFoundError(f'No results found in {MET}; run evaluator first.')
results=pd.DataFrame(records)
results.to_csv(OUT/'metrics_long.csv',index=False)
display(results[['condition','stress','n_correct','n_samples','accuracy','balanced_accuracy','precision_rotten','recall_rotten','f1_rotten']])


## 2. Clean-test and robustness comparison
All displayed accuracies are percentages. Missing cells remain NaN.

In [ ]:
def pivot(metric):
    return results.pivot(index='condition',columns='stress',values=metric).reindex(index=MODELS,columns=STRESSES)
accuracy=100*pivot('accuracy')
display(accuracy.round(2))
accuracy.to_csv(OUT/'accuracy_percent.csv')
clean=results.query('stress == "normal"').set_index('condition').reindex(MODELS)
cols=['accuracy','balanced_accuracy','precision_rotten','recall_rotten','f1_rotten']
display((100*clean[cols]).round(2))
(100*clean[cols]).to_csv(OUT/'clean_metrics_percent.csv')
fig,ax=plt.subplots(figsize=(10,5))
accuracy.plot.bar(ax=ax,rot=0); ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0,105); ax.legend(title='Illumination'); ax.grid(axis='y',alpha=.25)
save(fig,'accuracy_comparison.png')


## 3. Accuracy drop and ablation
ΔA = normal accuracy − stressed accuracy, in percentage points. Negative values indicate improvement. Compare absolute stressed accuracy as well as drops. E3−B1 tests the proposed extension against the Gaussian baseline; E3−E1 and E3−E2 isolate the added operation descriptively.

In [ ]:
drops=accuracy[STRESSES[1:]].rsub(accuracy['normal'],axis=0)
display(drops.round(2)); drops.to_csv(OUT/'accuracy_drop_pp.csv')
fig,ax=plt.subplots(figsize=(10,5)); drops.plot.bar(ax=ax,rot=0); ax.axhline(0,color='black',linewidth=.8); ax.set_ylabel('Accuracy drop (percentage points)'); ax.legend(title='Stress'); save(fig,'accuracy_drop.png')
comparisons=pd.DataFrame({f'E3 - {other}':accuracy.loc['E3']-accuracy.loc[other] for other in ['B0','B1','E1','E2']}).T
display(comparisons.round(2)); comparisons.to_csv(OUT/'ablation_gain_pp.csv')
mean_stressed=accuracy[STRESSES[1:]].mean(axis=1,skipna=False)
display(mean_stressed.rename('Mean stressed accuracy (%)').sort_values(ascending=False))


## 4. Confusion matrices and critical errors
Rows are actual classes, columns predicted classes. The bottom-left cell is rotten predicted fresh (false negative).

In [ ]:
fig,axes=plt.subplots(5,4,figsize=(13,15),squeeze=False)
for i,model in enumerate(MODELS):
    for j,stress in enumerate(STRESSES):
        ax=axes[i,j]; subset=results.query('condition == @model and stress == @stress')
        ax.set_title(f'{model} | {stress}',fontsize=10)
        if subset.empty: ax.text(.5,.5,'Not available',ha='center'); ax.set_axis_off(); continue
        cm=np.asarray(subset.iloc[0].confusion_matrix); ax.imshow(cm,vmin=0,vmax=160,cmap='Blues')
        for r in range(2):
            for k in range(2): ax.text(k,r,str(cm[r,k]),ha='center',va='center',color='white' if cm[r,k]>80 else 'black')
        ax.set_xticks([0,1],['Fresh','Rotten']); ax.set_yticks([0,1],['Fresh','Rotten']); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
fig.tight_layout(); save(fig,'confusion_matrices.png')
fn=pivot('rotten_misclassified_as_fresh'); display(fn); fn.to_csv(OUT/'rotten_as_fresh_counts.csv')
fig,ax=plt.subplots(figsize=(10,5)); fn.plot.bar(ax=ax,rot=0); ax.set_ylabel('Rotten predicted fresh (out of 160)'); ax.legend(title='Illumination'); save(fig,'rotten_as_fresh.png')


## 5. Paired prediction analysis
Match each prediction to the held-out manifest by path, then compare B1 versus E3 on identical stressed images. A rescue is B1 wrong and E3 correct; a regression is B1 correct and E3 wrong. Examples are illustrative, not independent proof of robustness.

In [ ]:
preds={}
for model,stress in [(m,s) for m in MODELS for s in STRESSES]:
    path=MET/(stem(model,stress)+'_predictions.csv')
    if not path.exists(): continue
    p=pd.read_csv(path)
    assert p.path.is_unique and set(p.path)==set(test.path), path
    p=p.merge(test[['path','label']],on='path',validate='one_to_one')
    assert p.true_label.eq(p.label).all() and p.predicted_label.isin([0,1]).all(),path
    assert p.correct.eq(p.true_label.eq(p.predicted_label).astype(int)).all(),path
    subset=results.query('condition == @model and stress == @stress')
    if not subset.empty: assert p.correct.sum()==subset.iloc[0].n_correct,path
    preds[model,stress]=p
pairs=[]
for stress in STRESSES[1:]:
    if ('B1',stress) not in preds or ('E3',stress) not in preds: continue
    a=preds['B1',stress]; b=preds['E3',stress]
    joined=a.merge(b[['path','predicted_label','probability_rotten','correct']],on='path',suffixes=('_B1','_E3'),validate='one_to_one')
    joined['stress']=stress
    joined['outcome']=np.select([(joined.correct_B1==0)&(joined.correct_E3==1),(joined.correct_B1==1)&(joined.correct_E3==0),(joined.correct_B1==1)&(joined.correct_E3==1)],['rescue','regression','both correct'],default='both wrong')
    pairs.append(joined)
paired=pd.concat(pairs,ignore_index=True) if pairs else pd.DataFrame()
if paired.empty: print('Paired predictions not yet available.')
else:
    display(pd.crosstab(paired.stress,paired.outcome)); paired.to_csv(OUT/'paired_B1_E3.csv',index=False)


## 6. Visual examples
Reconstruct the same resized RGB, stress and preprocessing pipeline. Choose the first sorted path for each available stress/outcome, with at most six examples. Do not change preprocessing based on these test examples.

In [ ]:
if not paired.empty:
    from PIL import Image
    from src.preprocess.stress import apply_stress
    from src.preprocess.pipeline import PIPELINES
    examples=(paired[paired.outcome.isin(['rescue','regression'])].sort_values(['stress','outcome','path']).groupby(['stress','outcome'],sort=False).head(1).head(6))
    if examples.empty: print('No rescue or regression examples found.')
    else:
        fig,axes=plt.subplots(len(examples),4,figsize=(12,3.2*len(examples)),squeeze=False)
        labels=['Fresh','Rotten']
        for i,(_,r) in enumerate(examples.iterrows()):
            with Image.open(ROOT/r.path) as im: rgb=np.array(im.convert('RGB').resize((250,250),Image.Resampling.BILINEAR))
            stressed=apply_stress(rgb,r.stress)
            images=[rgb,stressed,PIPELINES['B1'](stressed.copy()),PIPELINES['E3'](stressed.copy())]
            titles=[f'{r.fruit}: {labels[int(r.true_label)]}',f'{r.stress} | {r.outcome}',f'B1: {labels[int(r.predicted_label_B1)]}',f'E3: {labels[int(r.predicted_label_E3)]}']
            for ax,image,title in zip(axes[i],images,titles): ax.imshow(image); ax.set_title(title,fontsize=10); ax.axis('off')
        fig.tight_layout(); save(fig,'paired_examples.png')


## 7. Interpretation checklist
- Does E3 improve absolute stressed accuracy over B1? On which conditions?
- Is clean accuracy preserved?
- Does E3 reduce rotten-as-fresh errors?
- Are E1 or E2 stronger alone?
- Discuss failures without assuming improvement. One seed, synthetic illumination, and potential near-duplicates limit generalisation. File counts include exact duplicate copies within splits.

Optional W&B comparison: use the workspace to cross-check the same condition, stage, seed and stress; do not mix validation accuracy with held-out test accuracy. Local JSON and CSV are necessary for the paired analysis.